In [6]:
import requests
from constants import BINANCE_EXCHANGE_INFO




def get_exchangeInfo():

    response = requests.get(BINANCE_EXCHANGE_INFO, timeout=10)
    response.raise_for_status()

    data = response.json()

    return data




data = get_exchangeInfo()

In [7]:
print(data.keys())

dict_keys(['timezone', 'serverTime', 'optionContracts', 'optionAssets', 'optionSymbols', 'rateLimits'])


In [8]:
symbol = data["optionSymbols"][0]
symbol

{'expiryDate': 1790323200000,
 'filters': [{'filterType': 'PRICE_FILTER',
   'minPrice': '5.000',
   'maxPrice': '750.000',
   'tickSize': '5.000'},
  {'filterType': 'LOT_SIZE',
   'minQty': '0.01',
   'maxQty': '200',
   'stepSize': '0.01'}],
 'symbol': 'BTC-260925-145000-C',
 'side': 'CALL',
 'strikePrice': '145000.000',
 'underlying': 'BTCUSDT',
 'unit': 1,
 'liquidationFeeRate': '0.001900',
 'minQty': '0.01',
 'maxQty': '200',
 'initialMargin': '0.15000000',
 'maintenanceMargin': '0.07500000',
 'minInitialMargin': '0.10000000',
 'minMaintenanceMargin': '0.05000000',
 'priceScale': 3,
 'quantityScale': 2,
 'quoteAsset': 'USDT',
 'status': 'TRADING',
 'contractType': 'CRYPTO_OPTIONS',
 'underlyingType': 'CRYPTO'}

In [14]:
from dataclasses import dataclass
from typing import List


@dataclass
class Filter:
    filterType: str
    minPrice: str | None = None
    maxPrice: str | None = None
    tickSize: str | None = None
    minQty: str | None = None
    maxQty: str | None = None
    stepSize: str | None = None


@dataclass
class OptionContract:
    expiryDate: int
    filters: List[Filter]
    symbol: str
    side: str
    strikePrice: str
    underlying: str
    unit: int
    liquidationFeeRate: str
    minQty: str
    maxQty: str
    initialMargin: str
    maintenanceMargin: str
    minInitialMargin: str
    minMaintenanceMargin: str
    priceScale: int
    quantityScale: int
    quoteAsset: str
    status: str
    contractType: str
    underlyingType: str

In [ ]:
from datetime import datetime, timezone

from utils import split_datetime

def utc_timestamp_to_date(timestamp_ms):
    return datetime.fromtimestamp(
        timestamp_ms / 1000,
        tz=timezone.utc
    ).strftime("%y%m%d")

instruments = {}

for row in data["optionSymbols"]:
    expiry = split_datetime(row["expiryDate"]).date
    symbol = row["symbol"]
    side = row["side"]
    strike = row["strikePrice"]
    underlying = row["underlying"]
    quoteAsset = row["quoteAsset"]

    if underlying not in instruments:
        instruments[underlying] = {}
    if side not in instruments[underlying]:
        instruments[underlying][side] = {}
    if expiry not in instruments[underlying]:
        instruments[underlying][side][expiry] = {}
    if strike not in instruments[underlying][side][expiry]:
        instruments[underlying][side][expiry][strike] = OptionContract()

BTC-260925-145000-C


In [ ]:
from local import cashe

def get_exchangeInfo():
    response = requests.get(BINANCE_EXCHANGE_INFO, timeout=10)
    response.raise_for_status()

    data = response.json()

    optionSymbols = data["optionSymbols"]

    for row in optionSymbols:
        expiry = split_datetime(row["expiryDate"]).date
        symbol = row["symbol"]
        side = row["side"]
        strike = row["strikePrice"]
        underlying = row["underlying"]
        quoteAsset = row["quoteAsset"]

        filters = [
            Filter(**f)
            for f in row.get("filters", [])
        ]

        if underlying not in cashe.option_instruments:
            cashe.option_instruments[underlying] = {}

        if side not in cashe.option_instruments[underlying]:
            cashe.option_instruments[underlying][side] = {}

        if expiry not in cashe.option_instruments[underlying][side]:
            cashe.option_instruments[underlying][side][expiry] = {}

        if strike not in cashe.option_instruments[underlying][side][expiry]:
            cashe.option_instruments[underlying][side][expiry][strike] = OptionContract(
                expiry,
                Filter(filters),
                symbol,
                side,
                strike,
                underlying,
                row["unit"],
                row["liquidationFeeRate"],
                row["minQty"],
                row["maxQty"],
                row["initialMargin"],
                row["maintenanceMargin"],
                row["minInitialMargin"],
                row["minMaintenanceMargin"],
                row["priceScale"],
                row["quantityScale"],
                quoteAsset,
                row["status"],
                row["contractType"],
                row["underlyingType"],
            )


In [4]:
import requests

symbols = [
    "BTC-250228-125000-C",
    "BTC-250228-125000-P",
    "BTC-250328-88000-C",
    "BTC-250328-88000-P",
    "BTC-250425-87000-C",
    "BTC-250425-87000-P",
    "BTC-250627-108000-C",
    "BTC-250627-108000-P",
]

url = "https://eapi.binance.com/eapi/v1/klines"

for symbol in symbols:

    params = {
        "symbol": symbol,
        "interval": "1h",
        "limit": 5,
    }

    response = requests.get(
        url,
        params=params,
        timeout=10,
    )

    print(
        symbol,
        "->",
        response.status_code,
        response.text[:200],
    )

BTC-250228-125000-C -> 400 {"code":-1121,"msg":"The symbol is either in PENDING_TRADING status or does not exist."}
BTC-250228-125000-P -> 400 {"code":-1121,"msg":"The symbol is either in PENDING_TRADING status or does not exist."}
BTC-250328-88000-C -> 400 {"code":-1121,"msg":"The symbol is either in PENDING_TRADING status or does not exist."}
BTC-250328-88000-P -> 400 {"code":-1121,"msg":"The symbol is either in PENDING_TRADING status or does not exist."}
BTC-250425-87000-C -> 400 {"code":-1121,"msg":"The symbol is either in PENDING_TRADING status or does not exist."}
BTC-250425-87000-P -> 400 {"code":-1121,"msg":"The symbol is either in PENDING_TRADING status or does not exist."}
BTC-250627-108000-C -> 400 {"code":-1121,"msg":"The symbol is either in PENDING_TRADING status or does not exist."}
BTC-250627-108000-P -> 400 {"code":-1121,"msg":"The symbol is either in PENDING_TRADING status or does not exist."}


In [3]:
import requests

info_url = "https://eapi.binance.com/eapi/v1/exchangeInfo"

data = requests.get(info_url, timeout=10).json()

symbols = data["optionSymbols"]

# Pick the first currently listed BTC call
symbol = next(
    x["symbol"]
    for x in symbols
    if x["underlying"] == "BTCUSDT"
    and x["side"] == "CALL"
)

print("Testing symbol:", symbol)

url = "https://eapi.binance.com/eapi/v1/klines"

params = {
    "symbol": symbol,
    "interval": "1h",
    "limit": 5,
}

r = requests.get(url, params=params, timeout=10)

print("Status:", r.status_code)
print("Response:", r.text)

Testing symbol: BTC-260925-145000-C
Status: 200
Response: [[1788253200000,"5.000","5.000","5.000","5.000","0.00",1788256799999,"0.00000",0,"0.00","0.00000","0"],[1788256800000,"5.000","5.000","5.000","5.000","0.00",1788260399999,"0.00000",0,"0.00","0.00000","0"],[1788260400000,"5.000","5.000","5.000","5.000","0.00",1788263999999,"0.00000",0,"0.00","0.00000","0"],[1788264000000,"5.000","5.000","5.000","5.000","0.00",1788267599999,"0.00000",0,"0.00","0.00000","0"],[1788267600000,"5.000","5.000","5.000","5.000","0.00",1788271199999,"0.00000",0,"0.00","0.00000","0"]]


In [11]:
lll = [2, 21, 534, 64, 2]

sumx = str(lll).replace(",", "+").replace("[", "").replace("]", "").replace(" ", "")

print(eval(sumx)) 

623


In [14]:
smas = [32, 12]

fast, slow = sorted(smas)

slow

32